# Model A Neurons — Activation Strength in Phase 6

**Question**: Do the ~1,363 Model A neurons (layers 18–27) that were selected from the original geo+math data still show strong activation deltas in the **new** business_ethics and philosophy domains?

**Model A context**:  
- Ablation: layers 18–27, ~1,363 neurons, 0.257% of network  
- Effect: Δ = −9.81 pts (p < 0.001), 4× causally validated  
- These neurons were identified from Phase 4 (geo + math only)

**What we test here**:  
For each of the 4 domains × 2 conditions (money, reward), compute the delta (condition − neutral) specifically at the Model A neuron positions and compare to the background (non-Model-A neurons in the same layers).

## 0. Configuration

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import json, os
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────
# Phase 4 activations (geo + math) — used to validate signal is consistent
PHASE4_ACT_DIR = "/mnt/upschrimpf2/scratch/mahdipou/models/Anhedonic-AI/Experiment1/phase4/extraction/activations"

# Phase 6 activations (geo + math + business_ethics + philosophy)
PHASE6_ACT_DIR = "/mnt/upschrimpf2/scratch/mahdipou/models/Anhedonic-AI/Experiment1/phase6/activations"  # adjust if needed

# neurons_A.json — from model-A-18-27.py working directory
NEURONS_A_JSON = "/mnt/upschrimpf2/scratch/mahdipou/models/Anhedonic-AI/Experiment1/phase4/extraction/neurons_A.json"

# Model A target layers
MODEL_A_LAYERS = list(range(18, 28))   # 18–27 inclusive
NUM_LAYERS     = 28
INTER_DIM      = 18944

DOMAINS = {
    'geo':             {'phase': 4, 'dir': PHASE4_ACT_DIR},
    'math':            {'phase': 4, 'dir': PHASE4_ACT_DIR},
    'business_ethics': {'phase': 6, 'dir': PHASE6_ACT_DIR},
    'philosophy':      {'phase': 6, 'dir': PHASE6_ACT_DIR},
}
CONDITIONS = ['money', 'reward']

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {
    'model_a':     '#DD8452',
    'background':  '#4C72B0',
    'geo':         '#55A868',
    'math':        '#4C72B0',
    'business_ethics': '#C44E52',
    'philosophy':  '#8172B2',
    'money':       '#CCB974',
    'reward':      '#64B5CD',
}
print('Configuration loaded.')

## 1. Load Model A Neuron Map

In [ ]:
with open(NEURONS_A_JSON) as f:
    neuron_map_raw = json.load(f)

# {layer_int: [neuron_idx, ...]}
neuron_map = {int(k): v for k, v in neuron_map_raw.items()}

# Only keep layers in MODEL_A_LAYERS (sanity check)
neuron_map = {l: v for l, v in neuron_map.items() if l in MODEL_A_LAYERS}

total_model_a = sum(len(v) for v in neuron_map.values())
print(f"Model A neuron map loaded:")
print(f"  Layers with neurons : {sorted(neuron_map.keys())}")
print(f"  Total neurons       : {total_model_a:,}")
for l in MODEL_A_LAYERS:
    n = len(neuron_map.get(l, []))
    bar = '█' * min(n // 5, 40)
    print(f"  L{l:<4}: {n:>4} neurons  {bar}")

## 2. Load All Activations & Compute Deltas

In [ ]:
def load_mean(act_dir, condition, domain):
    """Load .pt file → mean over questions → numpy [num_layers, inter_dim]"""
    path = os.path.join(act_dir, f"{condition}_activations_{domain}.pt")
    data = torch.load(path, map_location='cpu')
    tensors = [v for v in data.values() if isinstance(v, torch.Tensor)]
    stacked = torch.stack(tensors).float()   # [Q, L, D]
    return stacked.mean(dim=0).numpy()       # [L, D]

# Load neutral + conditions for every domain
acts = {}  # acts[domain][condition] = [L, D] array
for domain, cfg in DOMAINS.items():
    d = cfg['dir']
    acts[domain] = {}
    acts[domain]['neutral'] = load_mean(d, 'neutral', domain)
    for cond in CONDITIONS:
        acts[domain][cond] = load_mean(d, cond, domain)
    print(f"  {domain}: loaded (shape {acts[domain]['neutral'].shape})")

# Compute deltas
deltas = {}  # deltas[domain][condition] = [L, D]
for domain in DOMAINS:
    deltas[domain] = {}
    for cond in CONDITIONS:
        deltas[domain][cond] = acts[domain][cond] - acts[domain]['neutral']

print("\nDeltas computed for all domain × condition pairs.")

## 3. Extract Delta Values — Model A Neurons vs Background

In [ ]:
def extract_model_a_deltas(delta_arr):
    """Return |delta| values at Model A neuron positions (L18–27 only)."""
    vals = []
    for layer, neurons in neuron_map.items():
        vals.extend(np.abs(delta_arr[layer, neurons]).tolist())
    return np.array(vals)

def extract_background_deltas(delta_arr):
    """Return |delta| values at non-Model-A positions within layers 18–27."""
    vals = []
    for layer in MODEL_A_LAYERS:
        all_neurons = set(range(INTER_DIM))
        model_a_neurons = set(neuron_map.get(layer, []))
        bg_neurons = np.array(sorted(all_neurons - model_a_neurons))
        vals.extend(np.abs(delta_arr[layer, bg_neurons]).tolist())
    return np.array(vals)

# Build summary table
rows = []
model_a_vals  = {}   # for later plots
bg_vals       = {}

for domain in DOMAINS:
    model_a_vals[domain]  = {}
    bg_vals[domain]       = {}
    for cond in CONDITIONS:
        d_arr = deltas[domain][cond]
        ma    = extract_model_a_deltas(d_arr)
        bg    = extract_background_deltas(d_arr)
        model_a_vals[domain][cond] = ma
        bg_vals[domain][cond]      = bg

        # t-test: are Model A deltas significantly larger than background?
        t_stat, p_val = stats.ttest_ind(ma, bg, equal_var=False, alternative='greater')
        fold = ma.mean() / bg.mean() if bg.mean() > 0 else np.nan
        rows.append({
            'Domain': domain, 'Condition': cond,
            'Model A mean |Δ|': f"{ma.mean():.6f}",
            'Background mean |Δ|': f"{bg.mean():.6f}",
            'Fold enrichment': f"{fold:.2f}×",
            't-stat': f"{t_stat:.2f}",
            'p-value': f"{p_val:.2e}",
            'Significant': '✓' if p_val < 0.001 else ('~' if p_val < 0.05 else '✗'),
        })

summary = pd.DataFrame(rows).set_index(['Domain', 'Condition'])
display(summary)

## 4. Mean |Δ| — Model A vs Background Across All Domains

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
domain_list = list(DOMAINS.keys())
x = np.arange(len(domain_list))
w = 0.35

for ax, cond in zip(axes, CONDITIONS):
    ma_means = [model_a_vals[d][cond].mean() for d in domain_list]
    bg_means = [bg_vals[d][cond].mean()      for d in domain_list]
    ma_sems  = [model_a_vals[d][cond].std() / np.sqrt(len(model_a_vals[d][cond])) for d in domain_list]
    bg_sems  = [bg_vals[d][cond].std()      / np.sqrt(len(bg_vals[d][cond]))       for d in domain_list]

    b1 = ax.bar(x - w/2, ma_means, w, yerr=ma_sems, capsize=4,
                color=COLORS['model_a'], label='Model A neurons', alpha=0.9)
    b2 = ax.bar(x + w/2, bg_means, w, yerr=bg_sems, capsize=4,
                color=COLORS['background'], label='Background (L18–27)', alpha=0.9)

    # Fold enrichment annotation
    for i, (ma, bg) in enumerate(zip(ma_means, bg_means)):
        fold = ma / bg if bg > 0 else 0
        ax.text(i, max(ma, bg) + max(ma_sems[i], bg_sems[i]) + 0.0002,
                f"{fold:.1f}×", ha='center', fontsize=9, color='#333333', fontweight='bold')

    # Shade new domains
    ax.axvspan(1.5, 3.5, color='#f0f0f0', alpha=0.6, zorder=0)
    ax.text(2.5, ax.get_ylim()[1] * 0.97 if ax.get_ylim()[1] > 0 else 0.001,
            'New domains', ha='center', fontsize=9, color='#888888', style='italic')

    ax.set_xticks(x)
    ax.set_xticklabels([d.replace('_', '\n') for d in domain_list], fontsize=10)
    ax.set_ylabel('Mean |Δ| activation')
    ax.set_title(f'{cond.capitalize()} condition', fontweight='bold', fontsize=13)
    ax.legend(fontsize=9)

plt.suptitle('Model A Neurons vs Background — Mean |Δ| per Domain\n'
             'Numbers above bars = fold enrichment', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('modelA_fig1_mean_delta.png', bbox_inches='tight')
plt.show()

## 5. Distribution of |Δ| — Model A vs Background (Violin)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9), sharey='row')

for row_i, cond in enumerate(CONDITIONS):
    for col_i, domain in enumerate(domain_list):
        ax = axes[row_i][col_i]
        ma = model_a_vals[domain][cond]
        bg = bg_vals[domain][cond]

        # Subsample background to keep plot fast
        rng = np.random.default_rng(42)
        bg_sample = rng.choice(bg, size=min(len(ma)*5, len(bg)), replace=False)

        data_plot = [bg_sample, ma]
        vp = ax.violinplot(data_plot, positions=[0, 1], showmedians=True,
                           showextrema=False)
        vp['bodies'][0].set_facecolor(COLORS['background'])
        vp['bodies'][1].set_facecolor(COLORS['model_a'])
        for b in vp['bodies']: b.set_alpha(0.75)
        vp['cmedians'].set_color('black')

        # 3σ line of background
        sigma3 = 3 * bg.std()
        ax.axhline(sigma3, color='red', linewidth=1, linestyle='--', alpha=0.7, label='3σ bg')

        pct_above = (ma > sigma3).mean() * 100
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Background', 'Model A'], fontsize=8)
        ax.set_title(f"{domain.replace('_','\n')}\n{pct_above:.1f}% above 3σ",
                     fontsize=9, fontweight='bold')
        if col_i == 0:
            ax.set_ylabel(f'{cond.capitalize()}\n|Δ| activation', fontsize=9)

plt.suptitle('Distribution of |Δ| at Model A Neurons vs Background\n'
             'Red dashed = 3σ threshold of background distribution',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('modelA_fig2_distributions.png', bbox_inches='tight')
plt.show()

## 6. Per-Layer Analysis Within L18–27

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for ax, cond in zip(axes, CONDITIONS):
    x_pos = np.arange(len(MODEL_A_LAYERS))
    width = 0.2
    offsets = np.linspace(-0.3, 0.3, len(domain_list))

    for offset, domain in zip(offsets, domain_list):
        d_arr = deltas[domain][cond]
        layer_means = []
        for layer in MODEL_A_LAYERS:
            neurons = neuron_map.get(layer, [])
            if neurons:
                layer_means.append(np.abs(d_arr[layer, neurons]).mean())
            else:
                layer_means.append(0.0)
        ax.bar(x_pos + offset, layer_means, width=width,
               color=COLORS[domain], alpha=0.85, label=domain.replace('_', ' '))

    ax.set_xticks(x_pos)
    ax.set_xticklabels([f'L{l}' for l in MODEL_A_LAYERS])
    ax.set_ylabel('Mean |Δ| at Model A neurons')
    ax.set_title(f'{cond.capitalize()} condition — Per-layer signal at Model A neuron positions',
                 fontweight='bold', fontsize=13)
    ax.legend(fontsize=9)

plt.suptitle('Per-Layer Model A Signal Across All 4 Domains', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('modelA_fig3_per_layer.png', bbox_inches='tight')
plt.show()

## 7. Percentile Rank — Where Do Model A Neurons Sit in the Full Delta Distribution?

In [ ]:
print("=" * 65)
print("PERCENTILE RANK of Model A neurons within full L18–27 distribution")
print("=" * 65)

rows_pct = []
for domain in domain_list:
    for cond in CONDITIONS:
        d_arr = deltas[domain][cond]
        ma    = model_a_vals[domain][cond]
        bg    = bg_vals[domain][cond]
        full  = np.concatenate([ma, bg])

        # What percentile is the median Model A neuron?
        pct_median = stats.percentileofscore(full, np.median(ma))
        pct_mean   = stats.percentileofscore(full, ma.mean())
        pct_above_90 = (ma > np.percentile(full, 90)).mean() * 100
        pct_above_95 = (ma > np.percentile(full, 95)).mean() * 100
        pct_above_99 = (ma > np.percentile(full, 99)).mean() * 100

        rows_pct.append({
            'Domain': domain, 'Condition': cond,
            'Median percentile': f"{pct_median:.1f}",
            'Mean percentile':   f"{pct_mean:.1f}",
            '% above p90':       f"{pct_above_90:.1f}%",
            '% above p95':       f"{pct_above_95:.1f}%",
            '% above p99':       f"{pct_above_99:.1f}%",
        })

pct_df = pd.DataFrame(rows_pct).set_index(['Domain', 'Condition'])
display(pct_df)

## 8. Heatmap — Mean |Δ| at Each Model A Layer × Domain

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, cond in zip(axes, CONDITIONS):
    mat = np.zeros((len(MODEL_A_LAYERS), len(domain_list)))
    for ci, domain in enumerate(domain_list):
        d_arr = deltas[domain][cond]
        for ri, layer in enumerate(MODEL_A_LAYERS):
            neurons = neuron_map.get(layer, [])
            if neurons:
                mat[ri, ci] = np.abs(d_arr[layer, neurons]).mean()

    im = ax.imshow(mat, aspect='auto', cmap='YlOrRd', interpolation='nearest')
    ax.set_xticks(range(len(domain_list)))
    ax.set_xticklabels([d.replace('_', '\n') for d in domain_list], fontsize=9)
    ax.set_yticks(range(len(MODEL_A_LAYERS)))
    ax.set_yticklabels([f'L{l}' for l in MODEL_A_LAYERS], fontsize=9)
    ax.set_title(f'{cond.capitalize()} — Mean |Δ| at Model A neurons', fontweight='bold')

    # Annotate cells
    for ri in range(len(MODEL_A_LAYERS)):
        for ci in range(len(domain_list)):
            ax.text(ci, ri, f"{mat[ri, ci]:.4f}", ha='center', va='center',
                    fontsize=7, color='black' if mat[ri, ci] < mat.max()*0.7 else 'white')

    plt.colorbar(im, ax=ax, label='Mean |Δ|')

    # Vertical line separating old from new domains
    ax.axvline(1.5, color='white', linewidth=2, linestyle='--')
    ax.text(0.95, -0.08, 'Phase 4', transform=ax.transAxes,
            ha='right', fontsize=8, color='#555555')
    ax.text(0.97, -0.08, 'Phase 6 new', transform=ax.transAxes,
            ha='right', fontsize=8, color='#C44E52')

plt.suptitle('Heatmap: Mean |Δ| at Model A Neuron Positions (L18–27)\n'
             'Dashed white line = old vs new domains',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('modelA_fig4_heatmap.png', bbox_inches='tight')
plt.show()

## 9. 3σ Threshold — How Many Model A Neurons Are "Active" per Domain?

In [ ]:
print("=" * 65)
print("MODEL A NEURONS ABOVE 3σ THRESHOLD (per domain × condition)")
print("=" * 65)

# For each domain/condition, threshold = 3σ of the GLOBAL delta array
rows_sig = []
for domain in domain_list:
    for cond in CONDITIONS:
        d_arr   = deltas[domain][cond]
        global_sigma3 = 3 * np.std(d_arr)
        ma      = model_a_vals[domain][cond]
        n_above = (ma > global_sigma3).sum()
        pct     = n_above / total_model_a * 100
        rows_sig.append({
            'Domain': domain, 'Condition': cond,
            '3σ threshold': f"{global_sigma3:.6f}",
            '# Model A neurons above 3σ': n_above,
            '% of Model A': f"{pct:.1f}%",
            'Total Model A': total_model_a,
        })

sig_df = pd.DataFrame(rows_sig).set_index(['Domain', 'Condition'])
display(sig_df)

# Bar plot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, cond in zip(axes, CONDITIONS):
    subset = sig_df.xs(cond, level='Condition')
    n_above = [int(sig_df.loc[(d, cond), '# Model A neurons above 3σ']) for d in domain_list]
    colors  = [COLORS[d] for d in domain_list]
    bars = ax.bar([d.replace('_','\n') for d in domain_list], n_above,
                  color=colors, alpha=0.85, edgecolor='white')
    for bar, cnt in zip(bars, n_above):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                str(cnt), ha='center', fontweight='bold')
    ax.axhline(total_model_a, color='grey', linewidth=1, linestyle='--',
               label=f'Total Model A ({total_model_a})')
    ax.set_ylabel('# neurons above 3σ')
    ax.set_title(f'{cond.capitalize()} condition', fontweight='bold', fontsize=13)
    ax.legend(fontsize=9)

plt.suptitle('Model A Neurons Exceeding 3σ Threshold by Domain', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('modelA_fig5_above3sigma.png', bbox_inches='tight')
plt.show()

## 10. Final Verdict

In [ ]:
print("=" * 65)
print("FINAL VERDICT — Are Model A neurons still strongly activated?")
print("=" * 65)

for cond in CONDITIONS:
    print(f"\n  Condition: {cond.upper()}")
    print(f"  {'Domain':<20} {'Fold enrichment':>16} {'% above 3σ':>12} {'Significant':>12}")
    print('  ' + '-' * 64)
    for domain in domain_list:
        ma      = model_a_vals[domain][cond]
        bg      = bg_vals[domain][cond]
        d_arr   = deltas[domain][cond]
        fold    = ma.mean() / bg.mean() if bg.mean() > 0 else np.nan
        thresh  = 3 * np.std(d_arr)
        pct_sig = (ma > thresh).mean() * 100
        t, p    = stats.ttest_ind(ma, bg, equal_var=False, alternative='greater')
        sig     = '✓ YES' if p < 0.001 else ('~ marginal' if p < 0.05 else '✗ NO')
        tag     = ' ← NEW' if domain in ['business_ethics', 'philosophy'] else ''
        print(f"  {(domain+tag):<26} {fold:>10.2f}×   {pct_sig:>8.1f}%   {sig:>12}")